# 📤 Serving — Export Gold → JSON Estático
**LogiLake | D'LOGIA** — Capa Serving del Medallion Architecture

Este notebook exporta las tablas Gold a archivos JSON/CSV que se
subirán al repositorio GitHub y serán consumidos por el dashboard
HTML desplegado en Netlify (sin backend activo).

Flujo:
1. Leer tablas Delta Gold
2. Exportar como JSON single-file a DBFS
3. Descargar manualmente desde Databricks Files → dashboard/data/

In [ ]:
import json
from pyspark.sql import functions as F
from datetime import datetime

GOLD_PATH    = "/FileStore/logilake/gold"
EXPORT_PATH  = "/FileStore/logilake/serving"

dbutils.fs.mkdirs(EXPORT_PATH)
print(f"Export path: {EXPORT_PATH}")

In [ ]:
def export_to_json(table_name: str, orient: str = "records") -> dict:
    """
    Lee una tabla Gold Delta y la escribe como JSON en DBFS.
    Retorna el path de descarga y el número de registros.
    """
    df = spark.read.format("delta").load(f"{GOLD_PATH}/{table_name}")
    
    # Convertir a Pandas para serialización limpia
    pdf = df.toPandas()
    
    # Serializar timestamps a string ISO
    for col in pdf.select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns:
        pdf[col] = pdf[col].astype(str)
    
    # JSON como lista de records
    records = pdf.to_dict(orient=orient)
    
    output = {
        "meta": {
            "table": table_name,
            "records": len(records),
            "exported_at": datetime.utcnow().isoformat(),
            "source": "logilake_gold_v1"
        },
        "data": records
    }
    
    # Escribir en DBFS
    json_str = json.dumps(output, ensure_ascii=False, indent=None)
    dbutils.fs.put(f"{EXPORT_PATH}/{table_name}.json", json_str, overwrite=True)
    
    print(f"✓ {table_name}.json  ({len(records)} registros, {len(json_str)/1024:.1f} KB)")
    return output

# Exportar todas las tablas Gold
gold_tables = [
    "kpi_global",
    "kpi_monthly",
    "kpi_category",
    "kpi_nps",
    "kpi_seller_state",
]

for table in gold_tables:
    try:
        export_to_json(table)
    except Exception as e:
        print(f"✗ Error exportando {table}: {e}")

print("\nExportación completada.")
print(f"Archivos en: {EXPORT_PATH}")

In [ ]:
# ── Listar archivos exportados ────────────────────────────────────────────────
files = dbutils.fs.ls(EXPORT_PATH)
print("Archivos JSON generados:")
for f in files:
    print(f"  {f.name:35s}  {f.size/1024:6.1f} KB")

print(f"\nPara descargar: Databricks → Data → DBFS → FileStore/logilake/serving/")
print("Luego copiar los JSON a: dashboard/data/")

In [ ]:
# ── Preview de un JSON exportado ──────────────────────────────────────────────
preview_raw = dbutils.fs.head(f"{EXPORT_PATH}/kpi_global.json")
preview = json.loads(preview_raw)
print("Meta:", json.dumps(preview["meta"], indent=2))
print("Datos:", json.dumps(preview["data"], indent=2))

In [ ]:
# ── Exportar también como CSV (opcional para Power BI) ────────────────────────
for table in gold_tables:
    try:
        df = spark.read.format("delta").load(f"{GOLD_PATH}/{table}")
        df.coalesce(1).write \
          .mode("overwrite") \
          .option("header", "true") \
          .csv(f"{EXPORT_PATH}/csv/{table}")
        print(f"✓ CSV: {table}")
    except Exception as e:
        print(f"✗ {table}: {e}")

print("CSVs disponibles para Power BI en:", f"{EXPORT_PATH}/csv/")